# Diabetic Retinopathy — Manual Evaluation Control Panel

This notebook **does not train models**. Evaluate one completed model run at a time and keep each evaluation's files separate. Supported models:

- `efficientnet` — unweighted EfficientNetB0 baseline
- `efficientnet_weighted` — class-weighted EfficientNetB0
- `densenet` — class-weighted DenseNet121
- `fusion` — initial unweighted EfficientNetB0 + DenseNet121 feature fusion

Use the official `validation` split for model selection. Keep `test` held out until the model choices are settled. Test evaluation requires explicit confirmation and repeated test evaluation on the same model/run is blocked by default. Comparisons must use saved evaluations from the same split and require explicit selections.

Full-data and fixed-manifest training variants are evaluated under separate keys: `efficientnet` vs `efficientnet_undersampled`, `efficientnet_weighted` vs `efficientnet_weighted_undersampled`, `densenet` vs `densenet_undersampled`, and `fusion` vs `fusion_undersampled`. Full-data runs use complete official validation/test splits. Undersampled runs use the fixed selected subsets from the same 70:10:20 manifest (expected validation=1,289 and test=2,578 examples). The evaluation output records the selected-split checksum. Comparing undersampled runs requires the same whole split-manifest checksum, which fixes train, validation and test examples together.


## 1. Environment setup

Run this once after restarting the `dr_tf` kernel. It configures the Windows Conda DLL path before TensorFlow is imported and enables GPU memory growth. This cell does not load or evaluate a model.

In [ ]:
from configs import tensorflow_setup as _tensorflow_setup
import sys
import numpy as np
import tensorflow as tf
_tensorflow_setup.configure_gpu_memory_growth(tf)

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("EfficientNetB0 API:", hasattr(tf.keras.applications, "EfficientNetB0"))
print("DenseNet121 API:", hasattr(tf.keras.applications, "DenseNet121"))

from scripts.evaluate import (
    evaluate_model,
    list_saved_evaluations,
    compare_evaluations,
    MODEL_DISPLAY_NAMES,
)
from scripts.utils import print_run_status

print("Evaluation import successful. No model has been loaded yet.")

## 2. Inspect each model's runs

Check that the run you intend to evaluate is marked `completed` and has its best-validation weights. This cell only reads run metadata/status; it does not evaluate a model.

In [ ]:
model_names = sorted(MODEL_DISPLAY_NAMES)
for selected_model in model_names:
    print("\n" + "=" * 72)
    print_run_status(selected_model)

## 3. Unweighted EfficientNetB0 — evaluate one run

Choose either `efficientnet` (full training split) or `efficientnet_undersampled` (fixed 70:10:20 subsets from `prepare_undersampled_split.ipynb`; training cap 2,500/class, validation/test reduced), then enter a specific **completed** run ID such as `run_001`. Choose `validation` during model selection. Choose `test` only after the model choices are settled and type `FINAL-TEST` when prompted. Every call writes a new evaluation folder; existing results are never overwritten.

In [ ]:
efficientnet_eval = None
model_choices = ("efficientnet", "efficientnet_undersampled")
print("Available keys:", model_choices)
model_name = input("Model key: ").strip()
if model_name not in model_choices:
    raise ValueError(f"Choose one of {model_choices}.")
run_id = input(f"Completed run ID for {model_name} (example run_001): ").strip()
split = input("Evaluation split [validation/test] (default validation): ").strip().lower() or "validation"
batch_text = input("Evaluation batch size [4]: ").strip()
batch_size = int(batch_text or 4)

if split == "test":
    final_test_choice = input("Test split is held out. Type FINAL-TEST to proceed: ").strip()
    if final_test_choice != "FINAL-TEST":
        print("Evaluation cancelled; test split was not evaluated.")
    else:
        efficientnet_eval = evaluate_model(
            model_name=model_name,
            run_id=run_id,
            split=split,
            batch_size=batch_size,
            confirm_test=True,
        )
elif split == "validation":
    efficientnet_eval = evaluate_model(
        model_name=model_name,
        run_id=run_id,
        split=split,
        batch_size=batch_size,
    )
else:
    raise ValueError("Split must be 'validation' or 'test'.")

## 4. Class-weighted EfficientNetB0 — evaluate one run

Choose either `efficientnet_weighted` (full training split) or `efficientnet_weighted_undersampled` (fixed 70:10:20 subsets from `prepare_undersampled_split.ipynb`; training cap 2,500/class, validation/test reduced), then enter a specific **completed** run ID such as `run_001`. Choose `validation` during model selection. Choose `test` only after the model choices are settled and type `FINAL-TEST` when prompted. Every call writes a new evaluation folder; existing results are never overwritten.

In [ ]:
weighted_efficientnet_eval = None
model_choices = ("efficientnet_weighted", "efficientnet_weighted_undersampled")
print("Available keys:", model_choices)
model_name = input("Model key: ").strip()
if model_name not in model_choices:
    raise ValueError(f"Choose one of {model_choices}.")
run_id = input(f"Completed run ID for {model_name} (example run_001): ").strip()
split = input("Evaluation split [validation/test] (default validation): ").strip().lower() or "validation"
batch_text = input("Evaluation batch size [4]: ").strip()
batch_size = int(batch_text or 4)

if split == "test":
    final_test_choice = input("Test split is held out. Type FINAL-TEST to proceed: ").strip()
    if final_test_choice != "FINAL-TEST":
        print("Evaluation cancelled; test split was not evaluated.")
    else:
        weighted_efficientnet_eval = evaluate_model(
            model_name=model_name,
            run_id=run_id,
            split=split,
            batch_size=batch_size,
            confirm_test=True,
        )
elif split == "validation":
    weighted_efficientnet_eval = evaluate_model(
        model_name=model_name,
        run_id=run_id,
        split=split,
        batch_size=batch_size,
    )
else:
    raise ValueError("Split must be 'validation' or 'test'.")

## 5. Class-weighted DenseNet121 — evaluate one run

Choose either `densenet` (full training split) or `densenet_undersampled` (fixed 70:10:20 subsets from `prepare_undersampled_split.ipynb`; training cap 2,500/class, validation/test reduced), then enter a specific **completed** run ID such as `run_001`. Choose `validation` during model selection. Choose `test` only after the model choices are settled and type `FINAL-TEST` when prompted. Every call writes a new evaluation folder; existing results are never overwritten.

In [ ]:
densenet_eval = None
model_choices = ("densenet", "densenet_undersampled")
print("Available keys:", model_choices)
model_name = input("Model key: ").strip()
if model_name not in model_choices:
    raise ValueError(f"Choose one of {model_choices}.")
run_id = input(f"Completed run ID for {model_name} (example run_001): ").strip()
split = input("Evaluation split [validation/test] (default validation): ").strip().lower() or "validation"
batch_text = input("Evaluation batch size [4]: ").strip()
batch_size = int(batch_text or 4)

if split == "test":
    final_test_choice = input("Test split is held out. Type FINAL-TEST to proceed: ").strip()
    if final_test_choice != "FINAL-TEST":
        print("Evaluation cancelled; test split was not evaluated.")
    else:
        densenet_eval = evaluate_model(
            model_name=model_name,
            run_id=run_id,
            split=split,
            batch_size=batch_size,
            confirm_test=True,
        )
elif split == "validation":
    densenet_eval = evaluate_model(
        model_name=model_name,
        run_id=run_id,
        split=split,
        batch_size=batch_size,
    )
else:
    raise ValueError("Split must be 'validation' or 'test'.")

## 6. Feature-fusion model — evaluate one run

Choose either `fusion` (full training split) or `fusion_undersampled` (fixed 70:10:20 subsets from `prepare_undersampled_split.ipynb`; training cap 2,500/class, validation/test reduced), then enter a specific **completed** run ID such as `run_001`. Choose `validation` during model selection. Choose `test` only after the model choices are settled and type `FINAL-TEST` when prompted. Every call writes a new evaluation folder; existing results are never overwritten.

In [ ]:
fusion_eval = None
model_choices = ("fusion", "fusion_undersampled")
print("Available keys:", model_choices)
model_name = input("Model key: ").strip()
if model_name not in model_choices:
    raise ValueError(f"Choose one of {model_choices}.")
run_id = input(f"Completed run ID for {model_name} (example run_001): ").strip()
split = input("Evaluation split [validation/test] (default validation): ").strip().lower() or "validation"
batch_text = input("Evaluation batch size [4]: ").strip()
batch_size = int(batch_text or 4)

if split == "test":
    final_test_choice = input("Test split is held out. Type FINAL-TEST to proceed: ").strip()
    if final_test_choice != "FINAL-TEST":
        print("Evaluation cancelled; test split was not evaluated.")
    else:
        fusion_eval = evaluate_model(
            model_name=model_name,
            run_id=run_id,
            split=split,
            batch_size=batch_size,
            confirm_test=True,
        )
elif split == "validation":
    fusion_eval = evaluate_model(
        model_name=model_name,
        run_id=run_id,
        split=split,
        batch_size=batch_size,
    )
else:
    raise ValueError("Split must be 'validation' or 'test'.")

## 7. List saved evaluations

This lists existing reports only. It does not re-evaluate or rank saved models automatically.

In [ ]:
import pandas as pd

saved_evaluations = list_saved_evaluations()
if saved_evaluations:
    display(pd.DataFrame(saved_evaluations))
else:
    print("No evaluation reports are saved yet.")

## 8. Compare explicitly selected evaluations

Select at least two saved reports by entering their exact model key, run ID, and evaluation ID. The script refuses to compare a mixture of `validation` and `test` reports. It does not sweep through all saved runs; only the reports you enter are compared. The comparison adds per-class recall columns because minority-grade performance matters for this imbalanced dataset.

In [ ]:
comparison_count = int(input("Number of saved evaluations to compare [4]: ").strip() or 4)
if comparison_count < 2:
    raise ValueError("Enter at least two evaluation reports to compare.")

selected_evaluations = []
for index in range(comparison_count):
    print(f"\nEvaluation {index + 1} of {comparison_count}")
    selected_evaluations.append({
        "model_name": input("Model key (see the full/undersampled keys listed above): ").strip(),
        "run_id": input("Run ID (example run_001): ").strip(),
        "evaluation_id": input("Evaluation ID (example eval_001): ").strip(),
    })

comparison_result = compare_evaluations(selected_evaluations)
display(pd.DataFrame(comparison_result["rows"]))
print("Comparison files:", comparison_result["output_dir"])

## Output organization

- Model-run metrics, class recalls, confusion matrix, and predictions: `results/evaluations/<model_name>/<run_id>/eval_NNN/`
- A comparison of only the selected evaluation reports: `results/comparisons/comparison_NNN/`

Metrics include accuracy, macro-F1, weighted-F1, class-wise precision/recall/F1/support, confusion matrix, and quadratic weighted kappa (QWK). Undersampled metrics record training, evaluation-subset, and whole-manifest checksums. The comparison function rejects reports from different manifests. `predictions.csv` contains the split's enumerated example index, true/predicted label, and five class probabilities. Dataset examples are not rebalanced during evaluation.
Evaluation metrics and comparison rows record each run’s training sampling mode and per-class cap. This makes full-data and undersampled training strategies distinguishable in the saved reports.
